# Risk Parity

05/04 showed that a 60/40 portfolio takes about 94% of its risk from equities. Risk parity turns the question around: choose
the weights so that every asset contributes the **same** share of risk. The result needs no expected returns at all, only the
covariance matrix, which 06/04 and 08/02 showed can be estimated reasonably well. This notebook defines equal risk contribution
(ERC), solves it as a convex optimization problem, places it between the minimum-variance and the 1/N portfolio, and examines
the leverage it needs to compete with a balanced portfolio.

## 0. Math link

- Risk contributions and Euler allocation from [05/04](../05_risk_measures/04_risk_contributions_and_euler_allocation.ipynb):
  $\mathrm{RC}_i = w_i(\Sigma w)_i/\sigma(w)$, summing to $\sigma(w)$.
- [Gradients](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/03_gradients.ipynb):
  the ERC weights are the minimum of a convex function whose gradient condition is exactly "equal contributions".
- Why the means are the problem: [08/01](./01_why_mean_variance_optimization_breaks.ipynb); covariance estimation:
  [08/02](./02_covariance_shrinkage_ledoit_wolf.ipynb).

## 1. Introduction

Risk parity became popular after 2008 as an alternative to 60/40: instead of letting equities dominate the risk, give bonds,
equities and other asset classes equal risk budgets, then lever the whole portfolio to the desired volatility. It avoids the
noisiest input (expected returns) and tends to be more diversified. It also relies on leverage and on bonds being a good
diversifier, which 2022 put to the test.

After this notebook you can:

- define equal-risk-contribution and risk-budgeting portfolios;
- solve the two-asset case by hand and the general case by convex optimization;
- relate ERC to inverse-volatility weights, the minimum-variance portfolio and 1/N;
- compare risk parity with mean-variance portfolios under estimation error;
- compute the leverage needed to reach a target volatility and its effect on return.

## 2. Theory

### Equal risk contribution

With $\sigma(w) = \sqrt{w^\top\Sigma w}$ and $\mathrm{RC}_i = w_i(\Sigma w)_i/\sigma$, the ERC portfolio solves

$$
\mathrm{RC}_i = \frac{\sigma(w)}{N} \quad \text{for all } i, \qquad w_i > 0, \quad \textstyle\sum_i w_i = 1
$$

More generally, **risk budgeting** sets $\mathrm{RC}_i = b_i\,\sigma(w)$ with budgets $b_i > 0$, $\sum b_i = 1$.

### Two assets

$\mathrm{RC}_1 = \mathrm{RC}_2$ gives $w_1^2\sigma_1^2 + w_1w_2\rho\sigma_1\sigma_2 = w_2^2\sigma_2^2 + w_1w_2\rho\sigma_1\sigma_2$, so $w_1\sigma_1 = w_2\sigma_2$:

$$
w_1 = \frac{1/\sigma_1}{1/\sigma_1 + 1/\sigma_2} = \frac{\sigma_2}{\sigma_1 + \sigma_2}
$$

independent of the correlation (05/04, exercise 4).

### Inverse volatility

$w_i \propto 1/\sigma_i$ is ERC when all pairwise correlations are equal; otherwise it is an approximation ("naive risk parity"). Assets that
are highly correlated with the rest get too much weight under inverse volatility, and ERC corrects for it.

### Solving ERC: a convex problem

Consider (Maillard, Roncalli and Teiletche 2010; Spinu 2013)

$$
\min_{y > 0}\ f(y) = \tfrac12 y^\top\Sigma y - \sum_i b_i\ln y_i
$$

$f$ is strictly convex (positive definite quadratic plus $-\ln$), so there is a unique minimum. Its gradient condition is

$$
(\Sigma y)_i - \frac{b_i}{y_i} = 0 \quad\Longleftrightarrow\quad y_i(\Sigma y)_i = b_i
$$

so the risk contributions of $y$ are proportional to $b$. Rescaling $w = y/\sum_j y_j$ keeps the proportions. With $b_i = 1/N$ this gives ERC.

### Between minimum variance and 1/N

Write the ERC condition as $w_i(\Sigma w)_i = c$ for all $i$. The minimum-variance portfolio has equal **marginal** risks $(\Sigma w)_i$; 1/N has equal
**weights**; ERC has equal **products**. Maillard, Roncalli and Teiletche show

$$
\sigma_{\text{GMV}} \le \sigma_{\text{ERC}} \le \sigma_{1/N}
$$

### When is ERC optimal?

The tangency portfolio satisfies $(\mu_i - r_f)/\mathrm{MR}_i = \mathrm{SR}$ for all $i$ (05/04). If all assets have the same Sharpe ratio and the same
correlation with each other, ERC coincides with the tangency portfolio. Risk parity is therefore a bet that, over the long run,
asset classes are compensated roughly in proportion to their risk.

### Leverage

An ERC portfolio of equities and bonds has low volatility, because bonds get most of the capital. To reach a target volatility
$\sigma^*$, hold leverage $k = \sigma^*/\sigma_{\text{ERC}}$ and borrow $k - 1$ at the financing rate $r_b$:

$$
E[r] = r_f + k\,(\mu_{\text{ERC}} - r_f) - (k - 1)(r_b - r_f)
$$

The Sharpe ratio is preserved only if $r_b = r_f$. Leverage also adds margin and liquidity risk.

## 3. Worked Examples

Swiss equities 16%, global equities 17%, Swiss bonds 5%; correlations 0.8 (equities), 0.1 and 0.0 (equity–bond); $r_f = 0.5\%$.

### Example 1: equities and bonds

Swiss equities and Swiss bonds: $w_E = 5/21 = 23.8\%$, $w_B = 76.2\%$.

$$
\sigma^2 = 0.238^2\cdot 0.0256 + 0.762^2\cdot 0.0025 + 2\cdot 0.238\cdot 0.762\cdot 0.1\cdot 0.16\cdot 0.05 = 0.00145 + 0.00145 + 0.00029 = 0.00319
$$

$\sigma = 5.65\%$. To reach 10% volatility: leverage $10/5.65 = 1.77$.

### Example 2: inverse volatility for three assets

$1/0.16 : 1/0.17 : 1/0.05 = 6.25 : 5.88 : 20$, sum 32.13: weights 19.5%, 18.3%, 62.2%. Because the two equity markets are 0.8 correlated with each
other, together they carry more than two thirds of the risk; ERC will give them less.

### Example 3: expected return of levered risk parity

The three-asset ERC portfolio (Python section) has $\mu_{\text{ERC}} = 3.50\%$ and $\sigma_{\text{ERC}} = 6.15\%$. Levered to the 9.70% volatility of a 30/30/40 portfolio,
$k = 9.70/6.15 = 1.58$; with financing at $r_f$: $0.5\% + 1.58\cdot 3.0\% = 5.23\%$, above the 4.85% of 30/30/40. If financing costs 1 point more than $r_f$:
$5.23\% - 0.58\cdot 1\% = 4.65\%$, below it.

## 4. Finance Application

### Where it matters

- **Multi-asset funds.** "All-weather" and risk parity funds allocate risk budgets across equities, bonds, commodities and
  inflation-linked bonds, then lever to a target volatility (typically 10–12%).
- **Strategic allocation.** Even without leverage, risk contributions are a better diversification check than capital
  weights; a risk-budget constraint can sit inside a traditional allocation.
- **Factor and manager allocation.** Equal risk budgets across factor strategies or managers avoid having one volatile
  strategy dominate.
- **Robustness.** Since ERC needs no expected returns, it changes slowly as the covariance estimate updates (exercise and
  Python section).

### The usual misreadings

- "Risk parity is low risk." The unlevered portfolio is; the levered fund targets equity-like volatility with a large bond
  position financed by borrowing.
- "Equal risk contributions mean equal losses in a crisis." Contributions are computed from the covariance matrix; when
  stock–bond correlation turns positive (2022), the bond leg loses together with equities and the realized risk is far
  higher than planned.
- "ERC is optimal." Only under equal Sharpe ratios and equal correlations; otherwise it is a robust heuristic.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(42)

assets3 = ["Swiss equities", "global equities", "Swiss bonds"]
mu3 = np.array([0.065, 0.07, 0.02])
vols3 = np.array([0.16, 0.17, 0.05])
corr3 = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma3 = np.outer(vols3, vols3) * corr3
rf = 0.005

### ERC by convex optimization

In [ ]:
def risk_contributions(w, Sigma):
    sigma = np.sqrt(w @ Sigma @ w)
    rc = w * (Sigma @ w) / sigma
    return sigma, rc


def risk_budget_weights(Sigma, budgets=None):
    # minimize 0.5 y'Sigma y - sum b_i log y_i over y > 0, then rescale to sum 1
    n = Sigma.shape[0]
    b = np.full(n, 1 / n) if budgets is None else np.asarray(budgets, dtype=float)
    f = lambda y: 0.5 * y @ Sigma @ y - b @ np.log(y)
    grad = lambda y: Sigma @ y - b / y
    y0 = 1 / np.sqrt(np.diag(Sigma))
    y0 = y0 / np.sqrt(y0 @ Sigma @ y0)
    res = minimize(f, y0, jac=grad, method="L-BFGS-B", bounds=[(1e-10, None)] * n, options={"ftol": 1e-15, "gtol": 1e-12})
    return res.x / res.x.sum()


def gmv(Sigma):
    x = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return x / x.sum()


def tangency(m, Sigma):
    x = np.linalg.solve(Sigma, m)
    return x / x.sum()


w_erc3 = risk_budget_weights(Sigma3)
sig, rc = risk_contributions(w_erc3, Sigma3)
print("ERC weights:", dict(zip(assets3, w_erc3.round(4))))
print("risk shares:", (rc / sig).round(6), f" volatility {sig:.2%}")

w_2 = risk_budget_weights(Sigma3[np.ix_([0, 2], [0, 2])])
print(f"two assets (Swiss equities, bonds): {w_2.round(4)}, closed form {0.05 / 0.21:.4f}")

### Comparing allocations

In [ ]:
def describe(w, Sigma, mu):
    sigma, rc = risk_contributions(w, Sigma)
    return {"vol": sigma, "exp. return": w @ mu, "Sharpe": (w @ mu - rf) / sigma,
            **{f"risk share {a}": r / sigma for a, r in zip(assets3, rc)}}

inv_vol = (1 / vols3) / (1 / vols3).sum()
port3 = {"1/N": np.full(3, 1 / 3), "60/40": np.array([0.3, 0.3, 0.4]), "inverse vol": inv_vol,
         "ERC": w_erc3, "min variance": gmv(Sigma3), "tangency": tangency(mu3 - rf, Sigma3)}
comp = pd.DataFrame({k: {**dict(zip([f"w {a}" for a in assets3], v)), **describe(v, Sigma3, mu3)} for k, v in port3.items()})
comp.style.format("{:.3f}")

ERC gives each equity market about 16% of capital and bonds 69%; each carries exactly a third of the risk. Inverse volatility gives
the equities more (19.5% and 18.3%) because it ignores their 0.8 correlation, so they carry 77% of its risk. Among these allocations
ERC has nearly the tangency Sharpe ratio (0.488 against 0.490) without using expected returns, and the tangency itself is close
to ERC in this universe because the Sharpe ratios of the three assets are similar.

### Eight asset classes under estimation error

The universe of 08/01. For 1,000 ten-year histories, estimate the inputs and build: 1/N, ERC, minimum variance, long-only and
unconstrained plug-in tangency. Evaluate each with the true parameters.

In [ ]:
assets8 = ["Swiss eq", "global eq", "EM eq", "Swiss small caps", "CHF bonds", "global bonds (hedged)", "CHF corporates", "Swiss real estate"]
N8 = 8
vols8 = np.array([0.16, 0.17, 0.22, 0.19, 0.05, 0.045, 0.06, 0.12])
mu8 = np.array([0.065, 0.07, 0.08, 0.075, 0.02, 0.022, 0.028, 0.05])
corr8 = np.eye(N8)
for (i, j), v in {(0, 1): 0.80, (0, 2): 0.65, (0, 3): 0.85, (1, 2): 0.75, (1, 3): 0.70, (2, 3): 0.60,
                  (4, 5): 0.75, (4, 6): 0.85, (5, 6): 0.70}.items():
    corr8[i, j] = corr8[j, i] = v
for i in range(4):
    for j in range(4, 7):
        corr8[i, j] = corr8[j, i] = 0.25 if j == 6 else 0.10
    corr8[i, 7] = corr8[7, i] = 0.5
for j in range(4, 7):
    corr8[j, 7] = corr8[7, j] = 0.3
Sigma8 = np.outer(vols8, vols8) * corr8
m8 = mu8 - rf

def sharpe8(w):
    return (w @ m8) / np.sqrt(w @ Sigma8 @ w)

def long_only_tangency(m, S):
    neg = lambda w: -(w @ m) / np.sqrt(w @ S @ w)
    r = minimize(neg, np.full(len(m), 1 / len(m)), method="SLSQP", bounds=[(0, 1)] * len(m),
                 constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}])
    return r.x

w_erc8_true = risk_budget_weights(Sigma8)
L8 = np.linalg.cholesky(Sigma8 / 12)
rows, W_erc_hist = [], []
for h in range(300):
    R = mu8 / 12 + rng.standard_normal((120, N8)) @ L8.T
    m_hat, S_hat = R.mean(axis=0) * 12 - rf, np.cov(R, rowvar=False) * 12
    w_e = risk_budget_weights(S_hat)
    W_erc_hist.append(w_e)
    rows.append({"1/N": sharpe8(np.full(N8, 1 / N8)), "ERC": sharpe8(w_e), "min variance": sharpe8(gmv(S_hat)),
                 "long-only tangency": sharpe8(long_only_tangency(m_hat, S_hat)),
                 "plug-in tangency": sharpe8(np.linalg.solve(S_hat, m_hat))})
oos = pd.DataFrame(rows)
W_erc_hist = np.array(W_erc_hist)
print(f"true max Sharpe {np.sqrt(m8 @ np.linalg.solve(Sigma8, m8)):.3f}; ERC with true Sigma {sharpe8(w_erc8_true):.3f}")
pd.DataFrame({"mean true SR": oos.mean(), "5% quantile": oos.quantile(0.05), "std": oos.std()}).round(3)

In [ ]:
pd.DataFrame({"ERC (true Sigma)": w_erc8_true, "ERC estimated: mean": W_erc_hist.mean(axis=0),
              "ERC estimated: std": W_erc_hist.std(axis=0)}, index=assets8).style.format("{:.1%}")

ERC with estimated covariances has an average true Sharpe ratio of 0.525, almost the 0.526 of ERC with the true matrix, and a 5%
quantile of 0.523: estimation error barely affects it, since its weights move by only 0.5–2 points across histories. It beats 1/N
(0.477), minimum variance (0.387), long-only tangency (0.467) and the plug-in tangency (0.281), and comes within 0.02 of the true
maximum (0.545).

### Volatility ordering

In [ ]:
vol8 = lambda w: np.sqrt(w @ Sigma8 @ w)
w_gmv8 = gmv(Sigma8)
print(f"min variance {vol8(w_gmv8):.2%} (long-short; weights {w_gmv8.min():.2f} to {w_gmv8.max():.2f})")
print(f"ERC          {vol8(w_erc8_true):.2%}")
print(f"1/N          {vol8(np.full(N8, 1 / N8)):.2%}")

### Leverage: risk parity vs a balanced portfolio

Three-asset ERC levered to the volatility of a 60/40 portfolio (30/30/40), with financing at $r_f$ and at $r_f + 1\%$.

In [ ]:
w_6040 = np.array([0.3, 0.3, 0.4])
sig_6040 = np.sqrt(w_6040 @ Sigma3 @ w_6040)
sig_erc = np.sqrt(w_erc3 @ Sigma3 @ w_erc3)
k = sig_6040 / sig_erc
mu_erc = w_erc3 @ mu3
lev = {}
for spread in [0.0, 0.01]:
    lev[f"levered ERC, financing rf + {spread:.0%}"] = rf + k * (mu_erc - rf) - (k - 1) * spread
print(f"60/40: vol {sig_6040:.2%}, expected return {w_6040 @ mu3:.2%}, Sharpe {(w_6040 @ mu3 - rf) / sig_6040:.3f}")
print(f"ERC unlevered: vol {sig_erc:.2%}, expected return {mu_erc:.2%}, Sharpe {(mu_erc - rf) / sig_erc:.3f}; leverage {k:.2f}")
for name, r in lev.items():
    print(f"{name}: expected return {r:.2%}, Sharpe {(r - rf) / sig_6040:.3f}")

Levered 1.58 times to the volatility of 30/30/40 and financed at the risk-free rate, ERC is expected to earn 5.23% against 4.85%. A
financing spread of 1 point reduces that to 4.65% and the Sharpe ratio from 0.488 to 0.428, below the balanced portfolio's 0.448.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
names = ["1/N", "60/40", "inverse vol", "ERC", "min variance"]
colors = ["tab:red", "tab:orange", "tab:blue"]
for j, name in enumerate(names):
    w = port3[name]
    s_, rc_ = risk_contributions(w, Sigma3)
    for offset, vals, alpha in [(-0.2, w, 0.45), (0.2, rc_ / s_, 0.9)]:
        bottom = 0.0
        for i in range(3):
            ax.bar(j + offset, vals[i], bottom=bottom, width=0.35, color=colors[i], alpha=alpha,
                   label=assets3[i] if (j == 0 and offset > 0) else None)
            bottom += vals[i]
    ax.text(j - 0.2, 1.02, "capital", ha="center", fontsize=8)
    ax.text(j + 0.2, 1.02, "risk", ha="center", fontsize=8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names)
ax.set_ylim(0, 1.1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Capital weights (light) and risk shares (dark)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
rhos = np.linspace(-0.5, 0.9, 50)
w_eq_erc, w_eq_gmv = [], []
for r in rhos:
    S2 = np.array([[0.16**2, r * 0.16 * 0.05], [r * 0.16 * 0.05, 0.05**2]])
    w_eq_erc.append(risk_budget_weights(S2)[0])
    w_eq_gmv.append(gmv(S2)[0])
axes[0].plot(rhos, w_eq_erc, label="ERC")
axes[0].plot(rhos, w_eq_gmv, label="minimum variance")
axes[0].axhline(0.5, color="grey", linestyle=":", label="1/N")
axes[0].set_title("Equity weight, equities (16%) and bonds (5%)")
axes[0].set_xlabel("equity-bond correlation")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].legend()

axes[1].boxplot([oos[c] for c in oos.columns], showfliers=False)
axes[1].set_xticks(range(1, len(oos.columns) + 1))
axes[1].set_xticklabels(oos.columns, rotation=20, ha="right")
axes[1].axhline(np.sqrt(m8 @ np.linalg.solve(Sigma8, m8)), color="black", linewidth=1, label="true maximum")
axes[1].set_title("True Sharpe ratio, inputs from 10 years (300 histories)")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
front = []
w_g, w_t = gmv(Sigma3), tangency(mu3 - rf, Sigma3)
for t in np.linspace(-0.5, 4.0, 300):
    w = (1 - t) * w_g + t * w_t                          # two-fund separation: frontier = combinations of GMV and tangency
    front.append((np.sqrt(w @ Sigma3 @ w), w @ mu3))
front = np.array(front)
ax.plot(front[:, 0], front[:, 1], color="grey", label="efficient frontier (unconstrained)")
s_t = np.sqrt(w_t @ Sigma3 @ w_t)
xs = np.linspace(0, 0.2, 50)
ax.plot(xs, rf + (w_t @ mu3 - rf) / s_t * xs, color="grey", linestyle=":", label="capital market line")
for name, mk in [("1/N", "o"), ("60/40", "s"), ("inverse vol", "D"), ("ERC", "*"), ("min variance", "^"), ("tangency", "P")]:
    w = port3[name]
    ax.plot(np.sqrt(w @ Sigma3 @ w), w @ mu3, mk, markersize=10, label=name)
ax.plot(sig_6040, lev["levered ERC, financing rf + 0%"], "*", markersize=14, color="black", label="ERC levered to 60/40 vol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_xlim(0, 0.2)
ax.set_ylim(0, 0.09)
ax.set_title("Allocations in risk-return space (true parameters)")
ax.set_xlabel("volatility")
ax.set_ylabel("expected return")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. Two assets with volatilities 20% and 8%. Compute the ERC weights and the portfolio volatility for correlations 0 and 0.5.
2. For exercise 1 with correlation 0, how much leverage is needed to reach 10% volatility?
3. A levered risk parity fund lost about 15% in a year in which both equities and bonds fell sharply. **Interpret** this loss in
   terms of the assumptions behind the portfolio.

### Intermediate

4. Verify $\sigma_{\text{GMV}} \le \sigma_{\text{ERC}} \le \sigma_{1/N}$ for 100 random covariance matrices of five assets (draw $\Sigma = AA^\top/5 + 0.01I$ with $A$ standard normal).
5. Risk budgeting for the eight asset classes: 60% of the risk to the four equity classes (15% each) and 40% to the other four (10% each).
   Compute the weights and check the contributions.

### Advanced

6. Show numerically that ERC equals the tangency portfolio when all assets have the same Sharpe ratio and equal pairwise
   correlations (five assets, different volatilities, correlation 0.3).
7. Correlation shock: raise the stock–bond correlations of the three-asset universe from (0.1, 0.0) to 0.6. Compute the volatility of the
   old ERC weights under the new covariance matrix, the new ERC weights, and the volatility a 10%-target levered fund would have had
   before rebalancing.

## 8. Solutions / Checks

In [ ]:
# 1. and 2.
for r in [0.0, 0.5]:
    S2 = np.array([[0.04, r * 0.2 * 0.08], [r * 0.2 * 0.08, 0.0064]])
    w = risk_budget_weights(S2)
    s2 = np.sqrt(w @ S2 @ w)
    print(f"1. rho {r}: weights {w.round(4)}, vol {s2:.2%}" + (f";  2. leverage for 10%: {0.10 / s2:.2f}" if r == 0 else ""))

# 4.
ok = 0
for i in range(100):
    A = rng.standard_normal((5, 5))
    S5 = A @ A.T / 5 + 0.01 * np.eye(5)
    v = lambda w: np.sqrt(w @ S5 @ w)
    ok += v(gmv(S5)) <= v(risk_budget_weights(S5)) + 1e-12 <= v(np.full(5, 0.2)) + 2e-12
print(f"4. ordering holds in {ok} of 100 cases")

# 5.
b8 = np.r_[np.full(4, 0.15), np.full(4, 0.10)]
w5 = risk_budget_weights(Sigma8, b8)
s5, rc5 = risk_contributions(w5, Sigma8)
print("5.", pd.DataFrame({"weight": w5, "risk share": rc5 / s5}, index=assets8).round(3).to_string())

In [ ]:
# 6.
v5 = np.array([0.05, 0.10, 0.15, 0.20, 0.25])
C5 = np.full((5, 5), 0.3) + 0.7 * np.eye(5)
S5 = np.outer(v5, v5) * C5
m5 = 0.4 * v5                                   # same Sharpe ratio 0.4
print(f"6. ERC {risk_budget_weights(S5).round(4)}\n   tangency {tangency(m5, S5).round(4)}")

# 7.
corr_new = corr3.copy()
corr_new[0, 2] = corr_new[2, 0] = corr_new[1, 2] = corr_new[2, 1] = 0.6
Sigma_new = np.outer(vols3, vols3) * corr_new
w_new = risk_budget_weights(Sigma_new)
vol_old_new = np.sqrt(w_erc3 @ Sigma_new @ w_erc3)
k10 = 0.10 / sig_erc
print(f"7. old ERC vol: before {sig_erc:.2%}, after the shock {vol_old_new:.2%}; new ERC weights {dict(zip(assets3, w_new.round(3)))}")
print(f"   levered fund (leverage {k10:.2f}) realized vol before rebalancing: {k10 * vol_old_new:.2%}")

**1.** $w_1 = 8/28 = 28.6\%$, $w_2 = 71.4\%$ for both correlations. Volatility $\sqrt{0.286^2\cdot 0.04 + 0.714^2\cdot 0.0064} = 8.08\%$ with $\rho = 0$, and $9.90\%$ with $\rho = 0.5$.

**2.** $10/8.08 = 1.24$.

**3.** Risk parity's equal risk budgets are computed from a covariance matrix in which bonds diversify equities (low or
negative correlation). When inflation drives both down together, the correlation turns positive, the large levered bond
position loses at the same time as equities, and the portfolio's realized volatility and loss are far above plan. Rising rates also
increase the cost of the leverage. The loss is not a failure of the arithmetic but of the assumption that the covariance matrix
is stable.

**4.** The ordering holds in every case.

**5.** The equity classes get small capital weights (their volatility is high) but 15% of the risk each; the bond classes get large
weights. The contributions match the budgets.

**6.** The two weight vectors coincide.

**7.** With stock–bond correlation 0.6 the old ERC portfolio's volatility rises from 6.15% to 7.58%. The new ERC weights move toward
equities (from 31% to 36% in total): the bond leg's contribution rose most, because bonds now co-move with the equity part of
the portfolio. A fund levered 1.63 times to target 10% on the old estimate would run at 12.3% until it rebalances and deleverages,
typically after the losses.

## 9. Conclusion

- ERC sets every risk contribution $w_i(\Sigma w)_i/\sigma$ equal; risk budgeting sets them to chosen shares.
- For two assets the ERC weights are inverse-volatility weights, independent of the correlation.
- ERC is the unique minimum of the convex function $\tfrac12 y^\top\Sigma y - \sum b_i\ln y_i$, rescaled.
- Its volatility lies between the minimum-variance and the 1/N portfolio; it equals the tangency portfolio under equal Sharpe ratios
  and equal correlations.
- Without expected returns it is robust to estimation error, but it is bond-heavy and needs leverage to reach equity-like
  returns, which makes it vulnerable to rising stock–bond correlation and financing costs.

Next: [08/04 Black–Litterman as Bayesian updating](./04_black_litterman_as_bayesian_updating.ipynb).